In [286]:
# This code extracts the sequence and chain informations according to the assigned tutorials  

from biopandas.pdb import PandasPdb
import numpy as np

import os

# Specify the directory where you want to save the files
output_directory = "../Outputs/data_manipulation"

# Create the directory if it doesn't exist
if not os.path.exists(output_directory):
    os.makedirs(output_directory)

In [287]:
# Initialize a new PandasPdb object and fetch the PDB file from rcsb.org
# ppdb = PandasPdb().fetch_pdb('1crn')
# ppdb = PandasPdb().fetch_pdb('2C0K')
ppdb = PandasPdb().fetch_pdb('2W5I')

# Display the type of information in each dataframe
for df_name in ppdb.df:
    print(f"Dataframe: {df_name}")
    print(ppdb.df[df_name].info())
    print(ppdb.df[df_name].head(), "\n")

# Optionally, display the column names for each dataframe
for df_name in ppdb.df:
    print(f"Columns in {df_name}:")
    print(ppdb.df[df_name].columns, "\n")

Dataframe: ATOM
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1826 entries, 0 to 1825
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   record_name     1826 non-null   object 
 1   atom_number     1826 non-null   int64  
 2   blank_1         1826 non-null   object 
 3   atom_name       1826 non-null   object 
 4   alt_loc         1826 non-null   object 
 5   residue_name    1826 non-null   object 
 6   blank_2         1826 non-null   object 
 7   chain_id        1826 non-null   object 
 8   residue_number  1826 non-null   int64  
 9   insertion       1826 non-null   object 
 10  blank_3         1826 non-null   object 
 11  x_coord         1826 non-null   float64
 12  y_coord         1826 non-null   float64
 13  z_coord         1826 non-null   float64
 14  occupancy       1826 non-null   float64
 15  b_factor        1826 non-null   float64
 16  blank_4         1826 non-null   object 
 17  segment_id      1

In [288]:
df_atoms = ppdb.df["ATOM"]
df_atoms

,record_name,atom_number,blank_1,atom_name,alt_loc,residue_name,blank_2,chain_id,residue_number,insertion,...,x_coord,y_coord,z_coord,occupancy,b_factor,blank_4,segment_id,element_symbol,charge,line_idx
0,ATOM,1,,N,,GLU,,A,2,,...,36.817,-17.878,25.672,1.0,49.75,,,N,NaN,781
1,ATOM,2,,CA,,GLU,,A,2,,...,35.849,-16.736,25.763,1.0,50.30,,,C,NaN,782
2,ATOM,3,,C,,GLU,,A,2,,...,35.686,-16.171,24.367,1.0,48.82,,,C,NaN,783
3,ATOM,4,,O,,GLU,,A,2,,...,36.686,-15.796,23.746,1.0,48.31,,,O,NaN,784
4,ATOM,5,,CB,,GLU,,A,2,,...,36.356,-15.636,26.729,1.0,50.01,,,C,NaN,785
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1821,ATOM,1823,,O,,VAL,,B,124,,...,3.674,-8.591,-3.097,1.0,35.62,,,O,NaN,2603
1822,ATOM,1824,,CB,,VAL,,B,124,,...,2.159,-5.916,-4.617,1.0,34.60,,,C,NaN,2604
1823,ATOM,1825,,CG1,,VAL,,B,124,,...,1.361,-4.636,-4.526,1.0,39.67,,,C,NaN,2605
1824,ATOM,1826,,CG2,,VAL,,B,124,,...,3.459,-5.653,-5.394,1.0,37.54,,,C,NaN,2606


In [289]:
df_others = ppdb.df["OTHERS"]
df_others

,record_name,entry,line_idx
0,HEADER,HYDROLASE 10...,0
1,TITLE,RNASE A-AP3A COMPLEX,1
2,COMPND,MOL_ID: 1;,2
3,COMPND,2 MOLECULE: RIBONUCLEASE PANCREATIC;,3
4,COMPND,"3 CHAIN: A, B;",4
...,...,...,...
858,CONECT,1888 1887 1889,2782
859,CONECT,1889 1888 1890,2783
860,CONECT,1890 1881 1884 1889,2784
861,MASTER,674 0 2 8 18 0 7 6 1...,2785


TUTORIAL 1:
Extract sequence from a PDB ATOM record and save to a fasta file format

In [290]:
# Extract the HEADER part of the PDB file
header_entry = df_others[df_others['record_name'] == 'HEADER']['entry'].values
print(header_entry)

# Pull the last string. Corresponds to the code of the protein
header_entry = header_entry[0]
code_name = header_entry.split()[-1]

print("\nCode:",code_name)

['    HYDROLASE                               10-DEC-08   2W5I']

Code: 2W5I


In [291]:
# Extract the COMPOUND part of the PDB file
compnd_entry = df_others[df_others['record_name'] == 'COMPND']['entry'].values

print(compnd_entry)

molecule_name = None
chains = None

# Iterate over each line in the array
# This section of code assumes there is only one set of chain and one sequence in the protein
# For multiple set of chains and sequences, this code can be modified by adding loops
for line in compnd_entry:
    # Check if the line contains the molecule name
    if 'MOLECULE' in line:
        molecule_name = line.split(':')[1].strip()
        molecule_name = molecule_name.replace(";","")
    # Check if the line contains chain information
    if 'CHAIN' in line:
        chains = (line.split(':')[1].strip())
        chains = chains.replace(";","")

print("\nMolecule Name: ", molecule_name)
print("Chains: ", chains)


['    MOL_ID: 1;' '   2 MOLECULE: RIBONUCLEASE PANCREATIC;'
 '   3 CHAIN: A, B;' '   4 SYNONYM: RIBONUCLEASE A, RNASE 1, RNASE A;'
 '   5 EC: 3.1.27.5']

Molecule Name:  RIBONUCLEASE PANCREATIC
Chains:  A, B


In [292]:
# Extract the SOURCE part of the PDB file
source_entry = df_others[df_others['record_name'] == 'SOURCE']['entry'].values
print(source_entry)

scientific_name = ""
taxid_name = ""

# Iterate over each line in the array
for line in source_entry:
    # Check if the line contains the scientific name
    if 'ORGANISM_SCIENTIFIC' in line:
        scientific_name = line.split(':')[1].strip()
        scientific_name = scientific_name.replace(";","")
    if 'ORGANISM_TAXID' in line:
        taxid_name = line.split(':')[1].strip()
        taxid_name = taxid_name.replace(";","")
        taxid_name = "(" + taxid_name + ")" 

print(f"\nOrganism Name: {scientific_name}")
print("Tax ID:" + taxid_name)

['    MOL_ID: 1;' '   2 ORGANISM_SCIENTIFIC: BOS TAURUS;'
 '   3 ORGANISM_COMMON: CATTLE;' '   4 ORGANISM_TAXID: 9913;'
 '   5 ORGAN: PANCREAS;' '   6 OTHER_DETAILS: SIGMA CHEMICAL CO.']

Organism Name: BOS TAURUS
Tax ID:(9913)


In [293]:
# Extract each residue name and convert them to one letter abbreviations of amino acids
amino_acid_sequence = ppdb.amino3to1().residue_name

# print("Amino Acid Sequence: ")
# print(''.join(amino_acid_sequence))

amino_acid_sequence2 = ppdb.amino3to1()
print(amino_acid_sequence2)

amino_acid_sequence = ''.join(amino_acid_sequence)
print(amino_acid_sequence)

     chain_id residue_name
0           A            E
9           A            T
16          A            A
21          A            A
26          A            A
...       ...          ...
1788        B            F
1799        B            D
1807        B            A
1812        B            S
1818        B            V

[247 rows x 2 columns]
ETAAAKFERQHMDSSTSAASSSNYCNQMMKSRNLTKDRCKPVNTFVHESLADVQAVCSQKNVACKNGQTNCYQSYSTMSITDCRETGSSKYPNCAYKTTQANKHIIVACEGNPYVPVHFDASVKETAAAKFERQHMDSSTSAASSSNYCNQMMKSRNLTKDRCKPVNTFVHESLADVQAVCSQKNVACKNGQTNCYQSYSTMSITDCRETGSSKYPNCAYKTTQANKHIIVACEGNPYVPVHFDASV


In [294]:
# Order:
# code_name
# chains
# molecule_name
# scientific_name
# taxid_name
# amino_acid_sequence

first_line = code_name + "|" + "Chains " + chains + "|" + molecule_name + "|" + scientific_name + " " + taxid_name
second_line = amino_acid_sequence

In [295]:
# Write sequences to a FASTA file

file_name = os.path.join(output_directory, f"tutorial_1.fasta")

with open(file_name, "w") as fasta_file:
    fasta_file.write(f">{first_line}\n")
    fasta_file.write(f"{second_line}\n")

TUTORIAL 2:
Extract the full sequence from the SEQRES record and save it to a fasta file format. 

In [296]:
seqres_entry = df_others[df_others['record_name'] == 'SEQRES']['entry'].values
seqres_entry

array(['   1 A  124  LYS GLU THR ALA ALA ALA LYS PHE GLU ARG GLN HIS MET',
       '   2 A  124  ASP SER SER THR SER ALA ALA SER SER SER ASN TYR CYS',
       '   3 A  124  ASN GLN MET MET LYS SER ARG ASN LEU THR LYS ASP ARG',
       '   4 A  124  CYS LYS PRO VAL ASN THR PHE VAL HIS GLU SER LEU ALA',
       '   5 A  124  ASP VAL GLN ALA VAL CYS SER GLN LYS ASN VAL ALA CYS',
       '   6 A  124  LYS ASN GLY GLN THR ASN CYS TYR GLN SER TYR SER THR',
       '   7 A  124  MET SER ILE THR ASP CYS ARG GLU THR GLY SER SER LYS',
       '   8 A  124  TYR PRO ASN CYS ALA TYR LYS THR THR GLN ALA ASN LYS',
       '   9 A  124  HIS ILE ILE VAL ALA CYS GLU GLY ASN PRO TYR VAL PRO',
       '  10 A  124  VAL HIS PHE ASP ALA SER VAL',
       '   1 B  124  LYS GLU THR ALA ALA ALA LYS PHE GLU ARG GLN HIS MET',
       '   2 B  124  ASP SER SER THR SER ALA ALA SER SER SER ASN TYR CYS',
       '   3 B  124  ASN GLN MET MET LYS SER ARG ASN LEU THR LYS ASP ARG',
       '   4 B  124  CYS LYS PRO VAL ASN THR PHE 

In [297]:
# Initialize an empty dictionary
result_dict = {}

# Process each string in the array
for item in seqres_entry:
    # Split the string into words
    words = item.split()
    
    # Extract the second letter to get the chain identifier "A"
    second_letter = words[1]
    
    # Extract the substring after the '46', which starts from the 4th word
    substring = ' '.join(words[3:])
    
    # Append the substring to the corresponding key in the dictionary
    if second_letter in result_dict:
        result_dict[second_letter].append(substring)
    else:
        result_dict[second_letter] = [substring]

print(result_dict)

# Initialize a new dictionary to store the merged strings
merged_dict = {}

# Iterate through each key in the dictionary
for key, strings in result_dict.items():
    # Merge all strings in the list into one string
    merged_string = ' '.join(strings)
    # Add the merged string to the new dictionary
    merged_dict[key] = merged_string

print(merged_dict)

{'A': ['LYS GLU THR ALA ALA ALA LYS PHE GLU ARG GLN HIS MET', 'ASP SER SER THR SER ALA ALA SER SER SER ASN TYR CYS', 'ASN GLN MET MET LYS SER ARG ASN LEU THR LYS ASP ARG', 'CYS LYS PRO VAL ASN THR PHE VAL HIS GLU SER LEU ALA', 'ASP VAL GLN ALA VAL CYS SER GLN LYS ASN VAL ALA CYS', 'LYS ASN GLY GLN THR ASN CYS TYR GLN SER TYR SER THR', 'MET SER ILE THR ASP CYS ARG GLU THR GLY SER SER LYS', 'TYR PRO ASN CYS ALA TYR LYS THR THR GLN ALA ASN LYS', 'HIS ILE ILE VAL ALA CYS GLU GLY ASN PRO TYR VAL PRO', 'VAL HIS PHE ASP ALA SER VAL'], 'B': ['LYS GLU THR ALA ALA ALA LYS PHE GLU ARG GLN HIS MET', 'ASP SER SER THR SER ALA ALA SER SER SER ASN TYR CYS', 'ASN GLN MET MET LYS SER ARG ASN LEU THR LYS ASP ARG', 'CYS LYS PRO VAL ASN THR PHE VAL HIS GLU SER LEU ALA', 'ASP VAL GLN ALA VAL CYS SER GLN LYS ASN VAL ALA CYS', 'LYS ASN GLY GLN THR ASN CYS TYR GLN SER TYR SER THR', 'MET SER ILE THR ASP CYS ARG GLU THR GLY SER SER LYS', 'TYR PRO ASN CYS ALA TYR LYS THR THR GLN ALA ASN LYS', 'HIS ILE ILE VAL ALA

In [298]:
# Mapping dictionary from three-letter to one-letter amino acid codes
three_to_one = {
    "ALA": "A", "ARG": "R", "ASN": "N", "ASP": "D", "CYS": "C",
    "GLU": "E", "GLN": "Q", "GLY": "G", "HIS": "H", "ILE": "I",
    "LEU": "L", "LYS": "K", "MET": "M", "PHE": "F", "PRO": "P",
    "SER": "S", "THR": "T", "TRP": "W", "TYR": "Y", "VAL": "V", "SEC": "U", "ACE": ""
}

# Function to convert three-letter codes to one-letter codes
def convert_to_one_letter(amino_acid_sequence):
    one_letter_sequence = ''.join(three_to_one[aa] for aa in amino_acid_sequence.split())
    return one_letter_sequence

# Convert the sequences in the dictionary
converted_amino_acids = {key: convert_to_one_letter(seq) for key, seq in merged_dict.items()}

# Print the converted sequences
for key, seq in converted_amino_acids.items():
    print(f"{key}: {seq}")

# Store the converted sequences in strings
# sequence_A = converted_amino_acids['A']
# sequence_B = converted_amino_acids['B']

merged_sequence = "" 

# for chain in converted_amino_acids:
#     merged_sequence += converted_amino_acids[chain]

seen_chains = set()

for chain in converted_amino_acids:
    if converted_amino_acids[chain] not in seen_chains:
        merged_sequence += converted_amino_acids[chain]
        seen_chains.add(converted_amino_acids[chain])

print(merged_sequence)    

A: KETAAAKFERQHMDSSTSAASSSNYCNQMMKSRNLTKDRCKPVNTFVHESLADVQAVCSQKNVACKNGQTNCYQSYSTMSITDCRETGSSKYPNCAYKTTQANKHIIVACEGNPYVPVHFDASV
B: KETAAAKFERQHMDSSTSAASSSNYCNQMMKSRNLTKDRCKPVNTFVHESLADVQAVCSQKNVACKNGQTNCYQSYSTMSITDCRETGSSKYPNCAYKTTQANKHIIVACEGNPYVPVHFDASV
KETAAAKFERQHMDSSTSAASSSNYCNQMMKSRNLTKDRCKPVNTFVHESLADVQAVCSQKNVACKNGQTNCYQSYSTMSITDCRETGSSKYPNCAYKTTQANKHIIVACEGNPYVPVHFDASV


In [299]:
file_name2 = os.path.join(output_directory, f"tutorial_2.fasta")

# Save the the all sequence at once
with open(file_name2, "w") as fasta_file:
    fasta_file.write(f">")
    fasta_file.write(f"{merged_sequence} ")

TUTORIAL 3:
Pick a PDB with multiple chains. Extract chains and save each chain to an individual PDB.

In [300]:
sequence = ppdb.amino3to1()
print(sequence)

for chain_id in sequence['chain_id'].unique():
    print('\nChain ID: %s' % chain_id)
    print(''.join(sequence.loc[sequence['chain_id'] == chain_id, 'residue_name']))
    
    sequence_by_chain = ''.join(sequence.loc[sequence['chain_id'] == chain_id, 'residue_name'])

    file_name3 = os.path.join(output_directory, f"tutorial_3_chain_{chain_id}.pdb")
    with open(file_name3, 'w') as pdb_file:
        pdb_file.write(f"Chain {chain_id}: \n{sequence_by_chain}")


     chain_id residue_name
0           A            E
9           A            T
16          A            A
21          A            A
26          A            A
...       ...          ...
1788        B            F
1799        B            D
1807        B            A
1812        B            S
1818        B            V

[247 rows x 2 columns]

Chain ID: A
ETAAAKFERQHMDSSTSAASSSNYCNQMMKSRNLTKDRCKPVNTFVHESLADVQAVCSQKNVACKNGQTNCYQSYSTMSITDCRETGSSKYPNCAYKTTQANKHIIVACEGNPYVPVHFDASV

Chain ID: B
KETAAAKFERQHMDSSTSAASSSNYCNQMMKSRNLTKDRCKPVNTFVHESLADVQAVCSQKNVACKNGQTNCYQSYSTMSITDCRETGSSKYPNCAYKTTQANKHIIVACEGNPYVPVHFDASV


In [303]:
def extract_chains(pdb_file):
    # Load the PDB file using PandasPdb
    # ppdb = PandasPdb().read_pdb(pdb_file)

    # Get the unique chain IDs
    chain_ids = ppdb.df['ATOM']['chain_id'].unique()

    # Iterate over each chain ID
    for chain_id in chain_ids:
        # Filter the dataframe for the current chain
        chain_df = ppdb.df['ATOM'][ppdb.df['ATOM']['chain_id'] == chain_id]

        print(chain_df)

        # Create a new PandasPdb object for the chain
        chain_ppdb = PandasPdb()
        chain_ppdb.df['ATOM'] = chain_df

        # Write the chain to a new PDB file
        output_file = f"{pdb_file.split('.')[0]}_chain_{chain_id}.pdb"
        chain_ppdb.to_pdb(path=output_file, records=None, gz=False, append_newline=True)

# Specify the PDB file
pdb_file = "your_pdb_file.pdb"

# Extract chains
extract_chains(pdb_file)

    record_name  atom_number blank_1 atom_name alt_loc residue_name blank_2  \
0          ATOM            1                 N                  GLU           
1          ATOM            2                CA                  GLU           
2          ATOM            3                 C                  GLU           
3          ATOM            4                 O                  GLU           
4          ATOM            5                CB                  GLU           
..          ...          ...     ...       ...     ...          ...     ...   
902        ATOM          903                 O                  VAL           
903        ATOM          904                CB                  VAL           
904        ATOM          905               CG1                  VAL           
905        ATOM          906               CG2                  VAL           
906        ATOM          907               OXT                  VAL           

    chain_id  residue_number insertion  ... x_coord